# PBIP-008 ? Rotaci?n proyectada

Trazabilidad de la Fase 2. La l?gica ejecutable reside en `Scripts/rotacion_proyectada/`; este notebook solo presenta y valida resultados agregados. No contiene datos personales ni modifica Power BI.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / 'Scripts').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from Scripts.rotacion_proyectada.run_backtesting import run_analysis

## Ejecuci?n ?nica de la fuente de verdad

In [ ]:
result = run_analysis(write_outputs=False)
print('Dataset:', result['version_dataset'])

## Controles de calidad, incluido C9 independiente

In [ ]:
display(result['quality'][['regla', 'estado', 'detalle']])

## Gate temporal final (holdout marzo?julio 2026)

In [ ]:
holdout = result['holdout'].copy()
for col in ['mae_seleccion', 'rmse_seleccion', 'mae_holdout', 'rmse_holdout', 'mae_naive_holdout', 'rmse_naive_holdout']:
    holdout[col] = (holdout[col] * 100).round(3)
display(holdout[['grupo_empresa', 'modelo_seleccionado', 'mae_seleccion', 'rmse_seleccion', 'mase_seleccion', 'mae_holdout', 'rmse_holdout', 'mae_naive_holdout', 'rmse_naive_holdout', 'modelo_final', 'decision_final']])

## Predicciones fuera de dominio

In [ ]:
invalid = result['prediction_validity'].query('n_negativas > 0 or n_superiores_100 > 0')
display(invalid[['grupo_empresa', 'modelo', 'n_predicciones', 'n_negativas', 'n_superiores_100', 'prediccion_min', 'prediccion_max']])

## Recomendaci?n agosto?diciembre de 2026

In [ ]:
forecast = result['forecast'].copy()
forecast['tasa_pct'] = (forecast['tasa_mensual_retiros'] * 100).round(3)
display(forecast[['grupo_empresa', 'periodo', 'tipo_registro', 'modelo', 'tasa_pct', 'tipo_banda', 'cobertura_historica_observada']])

In [ ]:
chart = forecast.dropna(subset=['tasa_mensual_retiros'])
for group, sub in chart.groupby('grupo_empresa'):
    plt.plot(sub['periodo'].astype(str), sub['tasa_mensual_retiros'] * 100, marker='o', label=group)
plt.ylabel('Tasa mensual de retiros (%)')
plt.xlabel('Periodo')
plt.title('PBIP-008 ? Forecast, baseline o referencia seg?n fiabilidad')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Conciliaci?n agregada Planta Personal vs. RETIROS

In [ ]:
display(pd.Series(result['reconciliation_overview'], name='valor').to_frame())
residual = result['reconciliation'].query('diferencia_residual_proxy != 0')
display(residual[['periodo', 'grupo_empresa', 'diferencia_residual_proxy']])

## Lectura metodol?gica

- El holdout final no participa en la selecci?n.
- Una tasa negativa invalida el m?todo; no existe un l?mite superior de 100 % documentado.
- Las bandas se denominan **Banda de incertidumbre aproximada (80 %)** y reportan su cobertura hist?rica observada; no son intervalos predictivos calibrados.
- `REFERENCIA_DESCRIPTIVA`, `BASELINE` y `SIN_FORECAST` no se presentan como forecast estad?stico.